# Customer Retention Intelligence — Feature Engineering

## Objective
Build **leakage-safe customer snapshots** for predicting `churn_next_30_days`:
using only information known as of a chosen snapshot date, can we predict
whether a customer will churn within the next 30 days?

This notebook builds a customer-level, time-aware feature table --
`data/processed/customer_features.parquet` -- from the validated Phase 3
datasets. It does **not** train any model; that is a later phase.

## Phase 3 recap
Phase 1-3 are already complete: five raw datasets were generated, validated,
and saved as clean Parquet files in `data/processed/`. This notebook builds
directly on that validated output (see `notebooks/01_data_ingestion.ipynb`).

## A note on the synthetic churn data
`churn_events.csv` was synthetically generated (Phase 2) using each
customer's *full* available behavioral history as an input to a churn-risk
score. That generation mechanism is not changed here, and it may encode
relationships (e.g. between long-run engagement and churn) that would not
necessarily exist in a real production dataset collected online. **The
feature engineering performed in this notebook is strictly leakage-safe on
its own terms** -- every feature only ever uses data on or before its
snapshot date -- but this caveat means later model performance on this data
should be read as a methodology demonstration, not as evidence of
real-world generalization.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

from src.features import feature_engineering as fe

PROJECT_ROOT


PosixPath('/Users/dhananjaymandal/customer-retention-intelligence')

## 2. Data Loading

Load the **processed** Phase 3 datasets (`data/processed/*.parquet`) --
already schema-validated, with correct datetime dtypes and intentional
missingness preserved -- rather than re-reading and re-parsing the raw CSVs.


In [2]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

customers = pd.read_parquet(PROCESSED_DIR / "customers.parquet")
transactions = pd.read_parquet(PROCESSED_DIR / "transactions.parquet")
usage = pd.read_parquet(PROCESSED_DIR / "usage.parquet")
support_tickets = pd.read_parquet(PROCESSED_DIR / "support_tickets.parquet")
churn_events = pd.read_parquet(PROCESSED_DIR / "churn_events.parquet")

for name, df in [
    ("customers", customers), ("transactions", transactions), ("usage", usage),
    ("support_tickets", support_tickets), ("churn_events", churn_events),
]:
    print(f"{name:16s} shape={df.shape}")


customers        shape=(10000, 7)
transactions     shape=(299921, 6)
usage            shape=(1246888, 7)
support_tickets  shape=(85577, 8)
churn_events     shape=(1929, 3)


## 3. Snapshot Definition

**Monthly, month-end snapshots** from `2025-01-31` through `2026-05-31`
(17 snapshot dates). The final snapshot is `2026-05-31` -- deliberately no
later than that, because the prediction horizon is 30 days and the
underlying data only extends to `2026-06-30`
(`2026-05-31 + 30 days = 2026-06-30` exactly).

**Eligibility** for a given snapshot:
- the customer must have already signed up (`signup_date <= snapshot_date`);
- the customer must **not** already have churned as of that date
  (`churn_date > snapshot_date`, or never churned at all).

A customer who already churned by a given snapshot is not a meaningful
"will they churn in the next 30 days?" question at that point, so those
rows are excluded entirely -- no post-churn prediction records.


In [3]:
spine = fe.create_prediction_snapshots(customers, churn_events)

print("Snapshot dates:")
for d in sorted(spine["snapshot_date"].unique()):
    print(" ", pd.Timestamp(d).date())

print()
print(f"Eligible (customer_id, snapshot_date) rows: {len(spine):,}")
print(f"Unique customers appearing in at least one snapshot: {spine['customer_id'].nunique():,}")


Snapshot dates:
  2025-01-31
  2025-02-28
  2025-03-31
  2025-04-30
  2025-05-31
  2025-06-30
  2025-07-31
  2025-08-31
  2025-09-30
  2025-10-31
  2025-11-30
  2025-12-31
  2026-01-31
  2026-02-28
  2026-03-31
  2026-04-30
  2026-05-31

Eligible (customer_id, snapshot_date) rows: 136,701
Unique customers appearing in at least one snapshot: 9,746


## 4. Leakage Prevention

This is the single most important rule in this notebook:

```text
FEATURES: event_date <= snapshot_date   (transactions, usage, support tickets)
TARGET:   snapshot_date < churn_date <= snapshot_date + 30 days
```

Every feature-building function in `src/features/feature_engineering.py`
filters its source table to `<date column> <= snapshot_date` *before*
computing anything -- never `>`. `churn_date` (and `churn_reason`) are used
**only** to build the target column, and are never merged into the feature
dataset itself. Section 11 below runs explicit, non-tautological checks
that prove this boundary is actually enforced on the real data (not just
correct by construction / assumption).


## 5. Customer / Static Features

Demographic and contract attributes (constant per customer) plus tenure,
which is a direct function of `snapshot_date - signup_date` -- both always
known as of the snapshot, never future information.


In [4]:
customer_features = fe.build_customer_features(spine, customers)
print(customer_features.shape)
display(customer_features.head())


(136701, 9)


,customer_id,snapshot_date,age,country,plan,contract_type,acquisition_channel,tenure_days,tenure_months
0,C000001,2025-05-31,69,United States,Basic,Monthly,Referral,21,0.69
1,C000001,2025-06-30,69,United States,Basic,Monthly,Referral,51,1.68
2,C000001,2025-07-31,69,United States,Basic,Monthly,Referral,82,2.69
3,C000001,2025-08-31,69,United States,Basic,Monthly,Referral,113,3.71
4,C000001,2025-09-30,69,United States,Basic,Monthly,Referral,143,4.70


## 6. Transaction / RFM Features

Recency, frequency, and monetary features over 30/60/90/180-day windows,
using only `transaction_date <= snapshot_date`. Counts/sums default to 0
when a customer had no transactions in a window (a genuine zero-activity
fact); averages/maximums are left as `NaN` when the window has zero
transactions, since "the average transaction was $0" would misrepresent
"there was no transaction to average."


In [5]:
transaction_features = fe.build_transaction_features(spine, transactions)
print(transaction_features.shape)
display(transaction_features.filter(regex="customer_id|snapshot_date|_30d|days_since").head())


(149983, 45)


,customer_id,txn_count_30d,txn_amount_sum_30d,txn_amount_mean_30d,txn_amount_max_30d,txn_success_count_30d,txn_failed_count_30d,txn_purchase_count_30d,txn_renewal_count_30d,txn_upgrade_count_30d,txn_failed_rate_30d,days_since_last_transaction,days_since_last_successful_transaction,snapshot_date
0,C000001,1.0,35.71,35.71,35.71,1.0,0.0,1.0,0.0,0.0,0.0,21,21.0,2025-05-31
1,C000004,1.0,33.51,33.51,33.51,1.0,0.0,0.0,1.0,0.0,0.0,25,25.0,2025-05-31
2,C000005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37,37.0,2025-05-31
3,C000006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62,62.0,2025-05-31
4,C000007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40,40.0,2025-05-31


## 7. Usage Features

Engagement aggregates over a recent 30-day window, the *previous* 30-day
window (30-60 days back), and a 90-day window, using only
`usage_date <= snapshot_date`. Trend features (`*_change_30d`,
`*_pct_change_30d`) compare recent vs. previous -- both windows are
historical relative to the snapshot, so a trend is not a leak.

`usage_missing_rate_*` reports the intentional ~3% synthetic missingness
rate observed within each window; missing measurements are **not** silently
treated as zero (pandas' `.mean()` already skips `NaN` when averaging).


In [6]:
usage_features = fe.build_usage_features(spine, usage)
print(usage_features.shape)
display(usage_features.filter(regex="customer_id|snapshot_date|recent30d|change_30d|days_since").head())


(151074, 33)


,customer_id,active_days_recent30d,avg_login_count_recent30d,avg_session_minutes_recent30d,avg_feature_usage_count_recent30d,avg_pages_viewed_recent30d,avg_active_minutes_recent30d,usage_missing_rate_recent30d,login_count_change_30d,login_count_pct_change_30d,session_minutes_change_30d,session_minutes_pct_change_30d,feature_usage_count_change_30d,feature_usage_count_pct_change_30d,active_minutes_change_30d,active_minutes_pct_change_30d,days_since_last_usage,snapshot_date
0,C000001,7.0,3.857143,24.602857,4.0,8.142857,15.638571,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2025-05-31
1,C000004,2.0,2.500000,43.220000,3.5,15.000000,18.035000,0.00,1.500000,150.000000,42.810,10441.463415,2.500000,250.000000,17.725000,5717.741935,21,2025-05-31
2,C000005,4.0,2.333333,26.482500,3.0,10.000000,18.136667,0.15,0.833333,55.555556,12.400,88.052547,2.000000,200.000000,9.916667,120.640714,1,2025-05-31
3,C000006,5.0,6.600000,70.788000,6.4,19.600000,47.376000,0.00,1.266667,23.750000,10.238,16.908340,1.733333,37.142857,3.322667,7.542373,0,2025-05-31
4,C000007,2.0,0.000000,4.055000,3.0,3.000000,2.370000,0.10,-1.000000,-100.000000,NaN,NaN,3.000000,NaN,-8.420000,-78.035218,1,2025-05-31


## 8. Support-Ticket Features

Ticket volume, quality (satisfaction, resolution time), and category
breakdowns over 30/90/180-day windows, using only
`created_date <= snapshot_date`. Categories are discovered dynamically from
the data (never hard-coded), so `Billing` and `Technical` -- the categories
most relevant to churn -- are included automatically alongside the rest.


In [7]:
support_features = fe.build_support_features(spine, support_tickets)
print(support_features.shape)
display(support_features.filter(regex="customer_id|snapshot_date|_30d|days_since").head())


(106911, 52)


,customer_id,ticket_count_30d,pending_ticket_count_30d,resolved_ticket_count_30d,closed_ticket_count_30d,high_critical_ticket_count_30d,avg_satisfaction_30d,min_satisfaction_30d,avg_resolution_hours_30d,max_resolution_hours_30d,category_account_count_30d,category_billing_count_30d,category_feature_request_count_30d,category_login_count_30d,category_other_count_30d,category_performance_count_30d,category_technical_count_30d,days_since_last_ticket,snapshot_date
0,C000004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200,2025-05-31
1,C000007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,210,2025-05-31
2,C000010,3.0,0.0,3.0,0.0,1.0,2.666667,2.0,37.32,61.59,1.0,0.0,2.0,0.0,0.0,0.0,0.0,5,2025-05-31
3,C000011,1.0,0.0,1.0,0.0,0.0,4.000000,4.0,39.52,39.52,1.0,0.0,0.0,0.0,0.0,0.0,0.0,26,2025-05-31
4,C000016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37,2025-05-31


## 9. Target Construction

```text
churn_next_30_days = 1  if  snapshot_date < churn_date <= snapshot_date + 30 days
churn_next_30_days = 0  otherwise
```

`churn_date` and `churn_reason` are used here and nowhere else -- they are
never merged into the feature dataset.


In [8]:
target = fe.build_target(spine)
print(target.value_counts())
print(f"Positive rate: {target.mean() * 100:.3f}%")


churn_next_30_days
0    135031
1      1670
Name: count, dtype: int64
Positive rate: 1.222%


## 10. Feature Dataset Inspection

In [9]:
features = fe.build_feature_dataset(customers, transactions, usage, support_tickets, churn_events)

print("Shape:", features.shape)
print("Columns:", len(features.columns))


Shape: (136701, 134)
Columns: 134


In [10]:
features.dtypes.value_counts()


float64           122
str                 5
int64               3
bool                3
datetime64[us]      1
Name: count, dtype: int64

In [11]:
missing_counts = features.isna().sum()
missing_pct = (missing_counts / len(features) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_pct", ascending=False)
print(f"{len(missing_summary)} of {len(features.columns)} columns have any missing values.")
display(missing_summary.head(20))


50 of 134 columns have any missing values.


,missing_count,missing_pct
min_satisfaction_30d,101426,74.20
avg_satisfaction_30d,101426,74.20
max_resolution_hours_30d,101375,74.16
avg_resolution_hours_30d,101375,74.16
min_satisfaction_90d,72886,53.32
avg_satisfaction_90d,72886,53.32
max_resolution_hours_90d,72818,53.27
avg_resolution_hours_90d,72818,53.27
min_satisfaction_180d,56772,41.53
avg_satisfaction_180d,56772,41.53


In [12]:
print("Target distribution:")
print(features["churn_next_30_days"].value_counts())
print(f"Positive class: {features['churn_next_30_days'].mean() * 100:.3f}%")


Target distribution:
churn_next_30_days
0    135031
1      1670
Name: count, dtype: int64
Positive class: 1.222%


In [13]:
snapshots_per_customer = features.groupby("customer_id")["snapshot_date"].size()
print(snapshots_per_customer.describe())


count    9746.000000
mean       14.026370
std         4.248807
min         1.000000
25%        11.000000
50%        17.000000
75%        17.000000
max        17.000000
Name: snapshot_date, dtype: float64


In [14]:
numeric_columns = features.select_dtypes(include=[np.number]).columns
display(features[numeric_columns].describe().T.head(20))


,count,mean,std,min,25%,50%,75%,max
age,136701.0,43.426800,15.127303,18.0,30.000000,43.00000,57.00000,69.00
tenure_days,136701.0,505.144322,301.290577,0.0,250.000000,488.00000,738.00000,1246.00
tenure_months,136701.0,16.594765,9.897823,0.0,8.210000,16.03000,24.24000,40.93
txn_count_30d,136701.0,1.399931,1.399389,0.0,0.000000,1.00000,2.00000,13.00
txn_amount_sum_30d,136701.0,79.496116,105.432085,0.0,0.000000,49.53000,109.51000,1245.27
txn_amount_mean_30d,96024.0,56.795836,36.145463,5.0,29.870000,41.47000,70.37625,210.59
txn_amount_max_30d,96024.0,62.315507,39.478124,5.0,33.320000,45.30000,78.81000,222.78
txn_success_count_30d,136701.0,1.330005,1.353229,0.0,0.000000,1.00000,2.00000,13.00
txn_failed_count_30d,136701.0,0.069926,0.267566,0.0,0.000000,0.00000,0.00000,4.00
txn_purchase_count_30d,136701.0,0.910805,1.072123,0.0,0.000000,1.00000,1.00000,9.00


## 11. Leakage Validation

Two layers of checks:

1. **Structural / target-correctness checks** (`validate_feature_dataset`):
   snapshot vs. signup ordering, no post-churn snapshots, the target exactly
   matches the business rule, no duplicate `(customer_id, snapshot_date)`
   pairs, no infinite values.
2. **Non-tautological leakage audit** (`run_leakage_audit`): for a random
   sample of snapshots, each recency feature (e.g.
   `days_since_last_transaction`) is independently recomputed straight from
   the raw table filtered to `<= snapshot_date`, and compared to the
   pipeline's own output. Critically, for customers who have events both
   before *and* after their snapshot, it also checks what the recency value
   **would have been** had future events leaked in -- and confirms the
   actual feature does not match that leaky value. This proves the cutoff
   is doing real work, not just checking a condition that is trivially true
   by construction.


In [15]:
structural_validation = fe.validate_feature_dataset(features, customers, churn_events)
structural_validation


{'passed': True,
 'issues': [],
 'snapshot_before_signup': 0,
 'post_churn_snapshots': 0,
 'target_mismatches': 0,
 'duplicate_customer_snapshot_pairs': 0,
 'infinite_value_count': 0}

In [16]:
leakage_audit = fe.run_leakage_audit(features, transactions, usage, support_tickets, sample_size=300)
leakage_audit


{'transactions': {'checked': 300,
  'mismatches': 0,
  'leakage_proof_found': True},
 'usage': {'checked': 300, 'mismatches': 0, 'leakage_proof_found': True},
 'support_tickets': {'checked': 300,
  'mismatches': 0,
  'leakage_proof_found': True}}

In [17]:
all_leakage_checks_passed = all(result["mismatches"] == 0 for result in leakage_audit.values())
any_leakage_proof_demonstrated = any(result["leakage_proof_found"] for result in leakage_audit.values())

print("All leakage checks passed (0 mismatches):", all_leakage_checks_passed)
print("Cutoff demonstrably changes the result vs. using full history:", any_leakage_proof_demonstrated)

assert structural_validation["passed"], "Structural validation failed -- see structural_validation['issues']"
assert all_leakage_checks_passed, "Leakage audit found mismatches -- a feature used future data"


All leakage checks passed (0 mismatches): True
Cutoff demonstrably changes the result vs. using full history: True


## 12. Save Feature Dataset

Saved to `data/processed/customer_features.parquet`. Raw CSVs in
`data/raw/` are never touched by this notebook.


In [18]:
output_path = fe.save_feature_dataset(features)
output_path


PosixPath('/Users/dhananjaymandal/customer-retention-intelligence/data/processed/customer_features.parquet')

## 13. Final Phase 4 Summary

In [19]:
num_features = features.shape[1] - 3  # exclude customer_id, snapshot_date, churn_next_30_days
positive_count = int(features["churn_next_30_days"].sum())
negative_count = int((features["churn_next_30_days"] == 0).sum())

summary = {
    "rows": len(features),
    "unique_customers": features["customer_id"].nunique(),
    "unique_snapshots": features["snapshot_date"].nunique(),
    "feature_columns": num_features,
    "positive_target_count": positive_count,
    "negative_target_count": negative_count,
    "positive_class_pct": round(positive_count / len(features) * 100, 3),
    "columns_with_missing_values": int((features.isna().sum() > 0).sum()),
    "structural_validation_passed": structural_validation["passed"],
    "leakage_audit_passed": all_leakage_checks_passed,
}

pd.Series(summary)


rows                            136701
unique_customers                  9746
unique_snapshots                    17
feature_columns                    131
positive_target_count             1670
negative_target_count           135031
positive_class_pct               1.222
columns_with_missing_values         50
structural_validation_passed      True
leakage_audit_passed              True
dtype: object